# Holdout Submission Builder

Combine holdout forecasts from each department into WMT submission-style CSVs (cases only, trucks only, and combined), then verify against `merged_data.csv` holdout rows.


In [1]:

import pandas as pd
from pathlib import Path

student_group = 13  # update if needed
models_dir = Path('models')

holdout_cases = []
holdout_trucks = []
for dept_dir in sorted(models_dir.glob('dept*')):
    dept = int(dept_dir.name.replace('dept',''))
    c_path = dept_dir / f'dept{dept}_holdout_cases.csv'
    t_path = dept_dir / f'dept{dept}_holdout_trucks.csv'
    if c_path.exists():
        cdf = pd.read_csv(c_path, parse_dates=['dt'])
        # standardize columns
        if 'predicted_cases' in cdf:
            cdf = cdf.rename(columns={'predicted_cases':'cases'})
        holdout_cases.append(cdf[['store_id','dept_id','dt','cases']])
    if t_path.exists():
        tdf = pd.read_csv(t_path, parse_dates=['dt'])
        if 'predicted_trucks' in tdf:
            tdf = tdf.rename(columns={'predicted_trucks':'trucks'})
        holdout_trucks.append(tdf[['store_id','dept_id','dt','trucks']])

cases_df = pd.concat(holdout_cases, ignore_index=True) if holdout_cases else pd.DataFrame()
trucks_df = pd.concat(holdout_trucks, ignore_index=True) if holdout_trucks else pd.DataFrame()
print('Cases rows:', len(cases_df), 'Trucks rows:', len(trucks_df))


Cases rows: 14000 Trucks rows: 14000


In [2]:

# Build submission-style files
cases_sub = cases_df.assign(student_group=student_group)[['student_group','dept_id','store_id','dt','cases']]
trucks_sub = trucks_df.assign(student_group=student_group)[['student_group','dept_id','store_id','dt','trucks']]
combined = cases_df.merge(trucks_df, on=['dept_id','store_id','dt'], how='outer')
combined = combined.assign(student_group=student_group)[['student_group','dept_id','store_id','dt','cases','trucks']]

out_dir = Path('submissions'); out_dir.mkdir(exist_ok=True)
cases_path = out_dir / 'submission_cases_holdout.csv'
trucks_path = out_dir / 'submission_trucks_holdout.csv'
combined_path = out_dir / 'submission_cases_trucks_holdout.csv'

cases_sub.sort_values(['dept_id','store_id','dt']).to_csv(cases_path, index=False)
trucks_sub.sort_values(['dept_id','store_id','dt']).to_csv(trucks_path, index=False)
combined.sort_values(['dept_id','store_id','dt']).to_csv(combined_path, index=False)

cases_sub.head(), trucks_sub.head(), combined.head()


(   student_group  dept_id  store_id         dt      cases
 0             13       41     10060 2025-08-18  48.198154
 1             13       41     10058 2025-08-18  54.762574
 2             13       41     10028 2025-08-18  44.351298
 3             13       41     10011 2025-08-18  41.408658
 4             13       41     10033 2025-08-18  43.700257,
    student_group  dept_id  store_id         dt    trucks
 0             13       41     10060 2025-08-18  2.028421
 1             13       41     10058 2025-08-18  2.123809
 2             13       41     10028 2025-08-18  1.998973
 3             13       41     10011 2025-08-18  2.003643
 4             13       41     10033 2025-08-18  2.011343,
    student_group  dept_id  store_id         dt      cases    trucks
 0             13        6     10001 2025-08-18  63.201769  2.887529
 1             13        6     10001 2025-08-19  68.692266  2.887529
 2             13        6     10001 2025-08-20  68.871955  2.887529
 3             13   

## Coverage check against merged_data holdout window

In [3]:

merged_path = Path('data/processed/merged_data.csv')
md = pd.read_csv(merged_path, parse_dates=['dt'])
# Holdout window
h_start = pd.Timestamp('2025-08-18')
h_end = pd.Timestamp('2025-09-14')
md_holdout = md[(md['dt']>=h_start) & (md['dt']<=h_end)]
print('Holdout rows in merged_data:', len(md_holdout))
print('Null cases:', md_holdout['cases'].isna().sum(), 'Null trucks:', md_holdout['trucks'].isna().sum())

# Merge predictions to see coverage
chk = md_holdout[['dept_id','store_id','dt','cases','trucks']].merge(
    combined[['dept_id','store_id','dt','cases','trucks']],
    on=['dept_id','store_id','dt'], how='left', suffixes=('_orig','_pred')
)
missing_cases = chk['cases_pred'].isna().sum()
missing_trucks = chk['trucks_pred'].isna().sum()
print('Missing predicted cases rows:', missing_cases)
print('Missing predicted trucks rows:', missing_trucks)

# Show any gaps
if missing_cases or missing_trucks:
    display(chk[chk['cases_pred'].isna() | chk['trucks_pred'].isna()].head())


Holdout rows in merged_data: 14000
Null cases: 14000 Null trucks: 14000
Missing predicted cases rows: 0
Missing predicted trucks rows: 0


## Notes
- Submission files are saved under `submissions/` with student_group set to 13 (change if needed).
- Coverage check ensures all holdout rows in merged_data.csv have predictions.